In [ ]:
apiKey='key'

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify
import google.generativeai as genai

# 1. Initialize Gemini API directly with your variable
# (Replace this placeholder with your NEW generated API key)

genai.configure(api_key=apiKey)

# 2. Fetch the target website
url = "https://xmindstudio.com/author/gulsahcakir"  # Example site
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    # 3. Clean up the HTML by turning it into readable Markdown
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Stripping scripts and styles helps save API context window space
    for script in soup(["script", "style"]):
        script.extract()
    
    markdown_content = markdownify(str(soup))

    # 4. Define the extraction prompt
    prompt = f"""
    You are an expert data extraction bot. Analyze the following webpage content in all the directories existing in the webiste and extract all information about background and work and connections and any information related to her in english. I do not need links only scrap all the existing links for that information.
    Return the data strictly as a clean JSON array of objects. Do not wrap the JSON in ```json markdown formatting.

    Webpage Content:
    {markdown_content}
    """

    # 5. Call the Gemini model
    model = genai.GenerativeModel("gemini-2.5-flash")
    print("Sending data to Gemini...")
    result = model.generate_content(prompt)
    
    # Print the clean structured data
    print("\n--- Extracted Data ---")
    print(result.text)

else:
    print(f"Failed to fetch webpage. Status code: {response.status_code}")

Sending data to Gemini...

--- Extracted Data ---
[
  {
    "name": "Gülşah ÇAKIR",
    "background": "Specific background details such as education or prior professional roles are not explicitly provided in the content. However, her prominent association with XMind Studio and extensive authorship of articles on investment and entrepreneurship topics imply a strong professional background and significant expertise in these fields.",
    "work": {
      "organization": "XMind Studio",
      "role": "Key figure / Contributor",
      "area_of_expertise": "Investment & Entrepreneurship Services, startup ecosystem analysis, fostering business collaborations, international company formation (e.g., Estonia's tax details), entrepreneurial analysis for investment decisions, venture capital fund strategies, entrepreneurial visas (e.g., Finland), and pitch presentation preparation for early-stage entrepreneurs. She also contributes to the formulation of XMind Studio's internal policies.",
      "

In [ ]:

print("Generating persona text...")
persona_response = model.generate_content(prompt)

# 2. Print it out to see it in the notebook
print("\n--- Generated Persona ---")
print(persona_response.text)

# 3. Use Python to save Gemini's response directly into a real .txt file
output_file = "persona.txt"
with open(output_file, "w", encoding="utf-8") as file:
    file.write(persona_response.text)

print(f"\n✅ Success! The persona has been physically saved as '{output_file}' in your directory.")

Generating persona text...

--- Generated Persona ---
**User Persona Profile: Gülşah ÇAKIR**

**NAME**
Gülşah ÇAKIR

**AFFILIATION**
XMind Studio

**ROLE**
Key individual and prolific author at XMind Studio, specializing in Investment & Entrepreneurship Services.

**BACKGROUND**
Gülşah Çakır demonstrates a strong professional background in the investment and entrepreneurship sectors, primarily through her association with XMind Studio. Her work involves both direct service provision and thought leadership, indicating deep expertise in startup ecosystems, business development, and related strategic business practices.

**EXPERIENCE & EXPERTISE**

*   **Business Leadership/Management:** Holds a leadership or principal role at XMind Studio, a firm dedicated to providing Investment & Entrepreneurship Services.
*   **Content Creation/Thought Leadership:** Authors numerous articles for the XMind Studio blog, sharing insights and expertise.
*   **Authored Topics Include:**
    *   Analysis of

In [18]:
# 5. Call the Gemini model
model = genai.GenerativeModel("gemini-2.5-flash")
print("Sending data to Gemini...")
result = model.generate_content(prompt)

# Clean up the response text just in case Gemini ignored the prompt 
# and added ```json ... ``` markdown code blocks anyway.
raw_text = result.text.strip()
if raw_text.startswith("```json"):
    raw_text = raw_text.split("```json")[1].split("```")[0].strip()
elif raw_text.startswith("```"):
    raw_text = raw_text.split("```")[1].split("```")[0].strip()

# 6. Parse and Save the JSON data
import json

try:
    # Convert the string from Gemini into a real Python JSON object
    json_data = json.loads(raw_text)
    
    # Define the output file name
    output_filename = "extracted_data.json"
    
    # Write it to a file with pretty indentation
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(json_data, f, ensure_ascii=False, indent=4)
        
    print(f"\n✅ Success! Data successfully saved to {output_filename}")
    
except json.JSONDecodeError as e:
    print("\n❌ Failed to parse response as JSON. Saving raw text to 'raw_output.txt' instead.")
    with open("raw_output2.txt", "w", encoding="utf-8") as f:
        f.write(result.text)

Sending data to Gemini...

✅ Success! Data successfully saved to extracted_data.json
